In [1]:
import duckdb
from functions.evaluation import evaluate
from networks.cnn_global_maxpooling import CNNModel
from networks.cnn_network import build_dataloaders


con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 98,530
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   0 | train: 7.7946 | val: 0.6910 | acc: 43.75% | AUC: 0.692  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   1 | train: 0.6344 | val: 0.5028 | acc: 93.12% | AUC: 0.818  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   2 | train: 0.5223 | val: 0.4099 | acc: 90.42% | AUC: 0.867  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   3 | train: 0.4591 | val: 0.3404 | acc: 95.28% | AUC: 0.878  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   4 | train: 0.4471 | val: 0.4703 | acc: 79.31% | AUC: 0.880  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling.pth
Epoch   5 | train

In [2]:
con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL
                 """).df()
con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 98,530
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling_fold1.pth
Epoch   0 | train: 6.5283 | val: 0.7135 | acc: 21.12% | AUC: 0.572  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling_fold1.pth
Epoch   1 | train: 0.6775 | val: 0.6540 | acc: 78.92% | AUC: 0.659  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling_fold1.pth
Epoch   2 | train: 0.5988 | val: 0.5719 | acc: 87.60% | AUC: 0.771  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling_fold1.pth
Epoch   3 | train: 0.5063 | val: 0.4854 | acc: 78.51% | AUC: 0.832  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_global_max_avg_std_pooling_fold1.pth
Epoch   4 | train: 0.4547 | val: 0.4510 | acc: 87.37% | AUC: 0.844  | LR: 0.00

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.084558,0.184757,94.521820,0.981365,0.949045,0.918519,1788,96,22,248
1,2,0.084316,0.104359,97.258364,0.990633,0.979809,0.922222,1844,38,21,249
2,3,0.062936,0.160517,96.703807,0.986022,0.974536,0.914498,1837,48,23,246
3,4,0.100800,0.213628,94.514179,0.979367,0.950053,0.910781,1788,94,24,245
4,5,0.073605,0.159623,96.935933,0.986300,0.977188,0.914498,1842,43,23,246
5,6,0.076264,0.138641,96.933086,0.986717,0.975571,0.925651,1837,46,20,249
6,7,0.085824,0.174826,96.610956,0.986518,0.974536,0.907063,1837,48,25,244
7,8,0.084926,0.203253,95.401765,0.978484,0.958068,0.925651,1805,79,20,249
8,9,0.120835,0.236145,96.888063,0.980138,0.980892,0.884758,1848,36,31,238
9,10,0.095655,0.178651,94.240595,0.978731,0.944268,0.929368,1779,105,19,250


In [3]:
test_rows = test_rows[test_rows['label'].isin([0,1])]
result = CNN.predict(test_rows)
_ = evaluate(result,threshold=0.5,proportion=70)

TypeError: evaluate() got an unexpected keyword argument 'threshold'